# 01 — Exploratory Data Analysis

Pure EDA on the **insurance quotes dataset** (~146 K rows, 25 columns).  
Goal: understand distributions, class balance, bind-rate drivers, date patterns, and correlations — **no feature engineering or modelling**.

In [ ]:
import warnings, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DATA_PATH = pathlib.Path("../datasets/insurance_quotes.csv")
SAVE_DIR  = pathlib.Path("../models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}):\n{list(df.columns)}\n")
df.head(3)

## 1. Data Quality

In [ ]:
print("=== Data Types ===")
print(df.dtypes.to_string(), "\n")

nulls = df.isnull().sum()
non_zero_nulls = nulls[nulls > 0]
print("=== Null Counts (non-zero only) ===")
print(non_zero_nulls.to_string() if len(non_zero_nulls) else "No nulls found")
print(f"\nTotal null values: {nulls.sum():,}")

dupes = df.duplicated().sum()
print(f"Duplicated rows:  {dupes:,}")

## 2. Target Variable — Policy Bind

In [ ]:
counts = df["Policy_Bind"].value_counts()
pcts   = df["Policy_Bind"].value_counts(normalize=True) * 100

print("=== Policy_Bind Distribution ===")
for label in counts.index:
    print(f"  {label}: {counts[label]:>7,}  ({pcts[label]:.2f}%)")

bind_rate = pcts.get("Yes", 0)
print(f"\nBind rate: {bind_rate:.2f}%")

fig, ax = plt.subplots(figsize=(5, 4))
colors = ["#e74c3c", "#2ecc71"]
counts.plot.bar(ax=ax, color=colors, edgecolor="white")
ax.set_title("Policy Bind Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("")
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f"{v:,}\n({pcts.iloc[i]:.1f}%)", ha="center", fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Numerical Feature Distributions

In [ ]:
df.describe()

num_cols = ["Quoted_Premium", "Driver_Age", "Driving_Exp",
            "HH_Vehicles", "HH_Drivers", "Prev_Accidents"]
palette  = ["#3498db", "#e67e22", "#9b59b6", "#1abc9c", "#e74c3c", "#f1c40f"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col, color in zip(axes.ravel(), num_cols, palette):
    ax.hist(df[col].dropna(), bins=40, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(col, fontsize=13, fontweight="bold")
    ax.set_ylabel("Frequency")

plt.tight_layout()
fig.savefig(SAVE_DIR / "eda_numerical_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Categorical Feature Distributions

In [ ]:
cat_cols = [
    "Agent_Type", "Region", "Policy_Type", "Coverage", "Veh_Usage",
    "Annual_Miles_Range", "Vehicl_Cost_Range", "Sal_Range",
    "Gender", "Marital_Status", "Education", "Re_Quote",
]

for col in cat_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_string(), "\n")

fig, axes = plt.subplots(3, 4, figsize=(22, 14))
for ax, col in zip(axes.ravel(), cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, palette="viridis")
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout()
fig.savefig(SAVE_DIR / "eda_categorical_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Bind Rate Analysis

In [ ]:
bind_flag = (df["Policy_Bind"] == "Yes").astype(int)

bind_cols = ["Agent_Type", "Region", "Coverage", "Sal_Range", "Veh_Usage", "Re_Quote"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.ravel(), bind_cols):
    rates = df.groupby(col)[["Policy_Bind"]].apply(
        lambda g: (g == "Yes").mean() * 100
    ).reset_index()
    rates.columns = [col, "Bind_Rate"]
    rates = rates.sort_values("Bind_Rate", ascending=False)
    sns.barplot(data=rates, x=col, y="Bind_Rate", ax=ax, palette="magma")
    ax.set_title(f"Bind Rate by {col}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Bind Rate (%)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    for bar in ax.patches:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%",
            ha="center", fontsize=8,
        )

plt.tight_layout()
fig.savefig(SAVE_DIR / "eda_bind_rate_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Date Analysis

In [ ]:
df["Q_Creation_DT"]  = pd.to_datetime(df["Q_Creation_DT"],  errors="coerce")
df["Q_Valid_DT"]     = pd.to_datetime(df["Q_Valid_DT"],     errors="coerce")
df["Policy_Bind_DT"] = pd.to_datetime(df["Policy_Bind_DT"], errors="coerce")

df["urgency_days"]  = (df["Q_Valid_DT"] - df["Q_Creation_DT"]).dt.days
df["days_to_bind"]  = (df["Policy_Bind_DT"] - df["Q_Creation_DT"]).dt.days

print("=== Urgency Days (quote validity window) ===")
print(df["urgency_days"].describe().to_string(), "\n")

bound = df[df["Policy_Bind"] == "Yes"]
print("=== Days to Bind (bound policies only) ===")
print(bound["days_to_bind"].describe().to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(df["urgency_days"].dropna(), bins=50, color="#2980b9", edgecolor="white")
ax1.set_title("Quote Validity Window (urgency_days)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Days")
ax1.set_ylabel("Frequency")

ax2.hist(bound["days_to_bind"].dropna(), bins=50, color="#27ae60", edgecolor="white")
ax2.set_title("Days to Bind (bound policies)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Days")
ax2.set_ylabel("Frequency")

plt.tight_layout()
fig.savefig(SAVE_DIR / "eda_date_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Correlation Analysis

In [ ]:
corr_cols = [
    "Prev_Accidents", "Prev_Citations", "Driver_Age", "Driving_Exp",
    "HH_Vehicles", "HH_Drivers", "Quoted_Premium",
]

corr_df = df[corr_cols].copy()
corr_df["Policy_Bind"] = (df["Policy_Bind"] == "Yes").astype(int)

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.5, ax=ax,
)
ax.set_title("Correlation Matrix (Numeric Features + Target)", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(SAVE_DIR / "eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Key Findings

In [ ]:
bind_binary = (df["Policy_Bind"] == "Yes").astype(int)

total_rows    = len(df)
bind_yes      = bind_binary.sum()
bind_no       = total_rows - bind_yes
overall_rate  = bind_yes / total_rows * 100
imbalance     = bind_no / bind_yes

total_nulls   = df.isnull().sum().sum()
null_col      = df.columns[df.isnull().any()].tolist()

region_rate   = df.groupby("Region")["Policy_Bind"].apply(lambda s: (s == "Yes").mean() * 100)
best_region   = region_rate.idxmax()
best_region_r = region_rate.max()

agent_rate    = df.groupby("Agent_Type")["Policy_Bind"].apply(lambda s: (s == "Yes").mean() * 100)
ea_rate       = agent_rate.get("EA", 0)
ia_rate       = agent_rate.get("IA", 0)

cov_rate      = df.groupby("Coverage")["Policy_Bind"].apply(lambda s: (s == "Yes").mean() * 100)
best_cov      = cov_rate.idxmax()
best_cov_r    = cov_rate.max()

print("=" * 60)
print("              KEY EDA FINDINGS")
print("=" * 60)
print(f"  Dataset size          : {total_rows:,} rows × {df.shape[1]} columns")
print(f"  Bind rate             : {overall_rate:.2f}% ({bind_yes:,} Yes / {bind_no:,} No)")
print(f"  Class imbalance ratio : 1 : {imbalance:.2f} (Yes : No)")
print(f"  Total null values     : {total_nulls:,}  (columns: {null_col})")
print()
print("  Key Patterns:")
print(f"    • Highest bind-rate region  : {best_region} ({best_region_r:.2f}%)")
print(f"    • EA bind rate              : {ea_rate:.2f}%")
print(f"    • IA bind rate              : {ia_rate:.2f}%")
print(f"    • Best-converting coverage  : {best_cov} ({best_cov_r:.2f}%)")
print("=" * 60)